# Baseline Dataset 구축: 전처리 및 데이터 증강

- **목적**: `train.csv`로부터 검증셋을 분리하고, 훈련셋을 클래스당 3,000건 수준으로 증강
- **참조**: `docs/strategy.md`, `plan_baseline_dataset.md`

In [ ]:
import pandas as pd
import numpy as np
import random
import re
import os
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

## 1. 데이터 로드 및 중복 제거

In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Step 1: 데이터 로드
raw_data = pd.read_csv('data/train.csv')
print(f"📦 [Original] 총 데이터 수: {len(raw_data)}")

# Step 1-1: 텍스트 정규화 (공백 제거 등)
# 앞뒤 공백 및 줄바꿈 내 연속 공백을 정리하여 중복 판정의 정확도를 높입니다.
raw_data['conversation'] = raw_data['conversation'].str.strip().replace(r'\s+', ' ', regex=True)

# Step 2: 완전 중복 제거
df = raw_data.drop_duplicates(subset=['conversation']).copy()
df = df.reset_index(drop=True)  # 인덱스 재설정 (준-중복 처리 시 필수)
exact_removed = len(raw_data) - len(df)
print(f"✅ [Exact Dedupe] 완전 중복 {exact_removed}건 제거 완료 (남은 데이터: {len(df)})")

# Step 3: 준-중복 제거 (유사도 0.95 이상)
tfidf = TfidfVectorizer().fit_transform(df['conversation'])
# 유사도 행렬 계산
cos_sim = cosine_similarity(tfidf, tfidf)

# 상삼각 행렬(Upper Triangle)만 탐색하여 자기 자신 및 중복 비교 방지
to_drop = set()
threshold = 0.95

for i in range(len(cos_sim)):
    if i in to_drop:
        continue
        
    # i번째 행에서 유사도가 threshold를 넘는 인덱스 추출
    similar_indices = np.where(cos_sim[i] > threshold)[0]
    
    for idx in similar_indices:
        if idx > i:  # 자기 자신보다 뒤에 있는 데이터만 제거 목록에 추가
            to_drop.add(idx)

# 최종 데이터 필터링
df_final = df.drop(index=list(to_drop)).reset_index(drop=True)
near_removed = len(to_drop)

print(f"✅ [Near Dedupe] 유사도 {threshold} 이상 준-중복 {near_removed}건 제거 완료")
print(f"🚀 [Final] 최종 학습 가용 데이터: {len(df_final)}건\n")

# 클래스별 분포 확인
print("📊 클래스별 데이터 분포:")
print(df_final['class'].value_counts())

# 검증을 위해 df_final 사용
df = df_final 


## 2. 검증셋(Validation Set) 분리 (Hold-out)

- 증강 전 원본 데이터에서 클래스별 200개(약 20%)를 미리 분리합니다.

In [ ]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split

# Step 1: 경로 확인 및 데이터 분할
if not os.path.exists('data'):
    os.makedirs('data')

# Stratified Split을 통해 클래스 비율을 유지하며 20% 분리
train_orig, val_holdout = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df['class']
)

# Step 2: 분할 결과 검증 및 출력
print("🛡️ [Validation Isolation] 검증셋 격리 완료")
print(f"  - 훈련 데이터(원본): {len(train_orig)}건")
print(f"  - 검증 데이터(순수): {len(val_holdout)}건 (전체의 20.0%)")

# 클래스별 분포가 일정한지 확인
val_dist = val_holdout['class'].value_counts()
print("\n📊 검증셋 클래스별 분포 (목표: 각 ~200건):")
print(val_dist)

# Step 3: Data Leakage 체크 (중복 여부 확인)
# 훈련 데이터와 검증 데이터 사이에 겹치는 conversation이 없는지 최종 확인
overlap = set(train_orig['conversation']) & set(val_holdout['conversation'])
if len(overlap) == 0:
    print("\n✅ [Leakage Check] 훈련셋과 검증셋 간 중복 문장 없음. 격리 성공!")
else:
    print(f"\n⚠️ [Warning] 중복 문장 {len(overlap)}건 발견. 확인이 필요합니다.")

# Step 4: 검증셋 저장 (불변 데이터)
val_holdout_path = 'data/val_holdout.csv'
val_holdout.to_csv(val_holdout_path, index=False)

print(f"\n💾 '{val_holdout_path}' 저장 완료.")
print("‼️ 주의: 이 데이터는 이후 어떠한 증강(Augmentation) 파이프라인에도 포함되지 않아야 합니다.")

# 이후 증강에는 train_orig만 사용
df_train = train_orig


## 3. 데이터 증강 (Augmentation) 함수 정의

In [ ]:
import random

# [전략 참조] strategy.md 및 plan_baseline_dataset.md 기반 보존 키워드
CORE_KEYWORDS = [
    "제발", "죽어", "죽여", "당장", "돈", "안돼", "부장님", "죄송", 
    "나오면", "다", "해", "맞을래", "조용히", "기다려", "진짜", "가지고"
]

# [추가] 일상적인 삽입 어구 (Random Insertion용)
FILLERS = ["진짜", "아니", "그러니까", "좀", "막", "근데"]

def sentence_shuffle(text):
    """[RS] 대화 내 문장 순서 swap (인접 발화 교체로 문맥 왜곡 최소화)"""
    sentences = text.split('\n')
    if len(sentences) < 3: # 너무 짧으면 셔플 의미 없음
        return text
    
    # 임의의 위치에서 인접한 두 문장 교체
    idx = random.randint(0, len(sentences) - 2)
    sentences[idx], sentences[idx+1] = sentences[idx+1], sentences[idx]
    return '\n'.join(sentences)

def random_deletion_safe(text, p=0.1):
    """[RD] 핵심 키워드를 보호하며 단어 무작위 삭제"""
    words = text.split()
    if len(words) < 10: # 너무 짧은 문장은 삭제 시 의미 훼손 큼
        return text
    
    new_words = []
    for word in words:
        # 1. 핵심 키워드가 포함되어 있는가? (예: '돈이' -> '돈' 포함)
        is_core = any(core in word for core in CORE_KEYWORDS)
        
        # 2. 키워드가 아니면서 확률 p에 걸리면 삭제
        if not is_core and random.random() < p:
            continue
        new_words.append(word)
        
    return ' '.join(new_words)

def random_insertion(text, p=0.1):
    """[RI] 일상적인 추임새나 강조어를 무작위 삽입"""
    words = text.split()
    if random.random() < p:
        insert_idx = random.randint(0, len(words))
        words.insert(insert_idx, random.choice(FILLERS))
    return ' '.join(words)

# [참고] SR(동의어 교체)은 KoNLPy 등의 형태소 분석기가 필요하므로 
# 구조만 잡아두고, 여기서는 가장 안정적인 RS, RD, RI를 통합합니다.

def augment_text(text):
    """[Dispatcher] 스텝 4의 통합 증강 파이프라인"""
    methods = [sentence_shuffle, random_deletion_safe, random_insertion]
    
    # 하나 또는 두 개의 기법을 무작위로 적용
    selected_methods = random.sample(methods, random.randint(1, 2))
    
    augmented_text = text
    for method in selected_methods:
        augmented_text = method(augmented_text)
        
    return augmented_text


## 4. 증강 실행 (목표: 클래스당 3,000건)

In [ ]:
import random
from tqdm import tqdm
import pandas as pd

# 모든 클래스(일반 대화 합성 포함)의 타겟 수량을 3,000건으로 통일
TARGET_COUNT = 3000 
augmented_records = []

print(f"🚀 [Augmentation Start] 모든 클래스 타겟: {TARGET_COUNT}건 (Perfect Balance)")

# train_orig에 있는 위협 4개 클래스에 대해 증강 수행
for label in train_orig['class'].unique():
    class_df = train_orig[train_orig['class'] == label]
    current_count = len(class_df)
    
    # 1. 원본 데이터(Hold-out 제외된 약 800건) 추가
    for _, row in class_df.iterrows():
        augmented_records.append({
            'class': row['class'], 
            'conversation': row['conversation'],
            'is_augmented': False
        })
    
    # 2. 증강 필요 수량 계산 (약 2,200건 추가 증강 필요)
    augment_needed = TARGET_COUNT - current_count
    factor = TARGET_COUNT / current_count
    
    print(f"  - 클래스 [{label}]: {current_count} -> {TARGET_COUNT} (증강 배율: {factor:.2f}x)")
    
    # 3. 증강 루프
    with tqdm(total=augment_needed, desc=f"   Augmenting {label}") as pbar:
        while augment_needed > 0:
            sample = class_df.sample(n=1).iloc[0]
            
            # 키워드 보호 로직이 담긴 augment_text 사용
            new_text = augment_text(sample['conversation'])
            
            # 중복 방지
            if new_text == sample['conversation']:
                continue
                
            augmented_records.append({
                'class': sample['class'], 
                'conversation': new_text,
                'is_augmented': True
            })
            augment_needed -= 1
            pbar.update(1)

# 데이터 병합 및 무작위 셔플
augmented_df = pd.DataFrame(augmented_records).sample(frac=1, random_state=42).reset_index(drop=True)

print(f"\n✨ [Completion] 위협 데이터 증강 완료!")
print(f"  - 현재 확보된 데이터: {len(augmented_df)}건 (4개 클래스)")
print(augmented_df['class'].value_counts())

print(f"\n💡 알림: 이후 추가될 '일반 대화(합성)' 역시 {TARGET_COUNT}건을 생성하여 밸런스를 맞추면 됩니다.")


## 5. 최종 데이터 저장

In [ ]:
import os

# 1. 저장 경로 확인
if not os.path.exists('data'):
    os.makedirs('data')

# 2. 분석용 컬럼(is_augmented) 제외하고 최종 필요한 컬럼만 추출 + 인덱스 부여
final_output = augmented_df[['class', 'conversation']].copy()
final_output.insert(0, 'idx', range(len(final_output)))

# 3. 최종 저장 (utf-8-sig는 엑셀에서 열었을 때 한글 깨짐을 방지합니다)
save_path = 'data/train_augmented_v1.csv'
final_output.to_csv(save_path, index=False, encoding='utf-8-sig')

print(f"✅ [Save Success] 총 {len(final_output)}건의 데이터가 '{save_path}'에 저장되었습니다.")
